<a href="https://colab.research.google.com/github/Shashankvuddagiri/Visual-Recognition-and-Scene-Understanding/blob/main/VRSU_Transfer_Learning_Lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Transfer Learning with MobileNetV2
# Oxford-IIIT Pets Dataset
# Prediction using USER UPLOADED IMAGE
# ==========================================

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import cv2
import gc

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from google.colab import files

# --------- Clear memory ---------
tf.keras.backend.clear_session()
gc.collect()

# --------- Load Dataset (only for training) ---------
(ds_train, ds_test), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
IMG_SIZE = 128
BATCH_SIZE = 16

# --------- Preprocessing function ---------
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)
    return image, label

train_ds = ds_train.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --------- Load MobileNetV2 (Pretrained) ---------
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False   # Freeze feature extractor

# --------- Add Classification Head ---------
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation="relu")(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# --------- Compile ---------
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# --------- Train Model ---------
model.fit(train_ds, epochs=3)

# --------- Evaluate (optional) ---------
loss, acc = model.evaluate(test_ds)
print("Validation Accuracy:", acc)

